# 01 — Data Ingestion
Parses VCF / VCF.bgz files into a DataFrame using `src/ingest.py` and saves to parquet for downstream notebooks.

In [9]:
import sys, os
from pathlib import Path

import pandas as pd

ROOT = Path(os.getcwd()).parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from ingest import parse_vcf_to_dataframe

In [10]:
# ── Configure ──────────────────────────────────────────────────────────────
VCF_FILES = [
    ROOT / 'data' / 'raw' / 'gnomad.genomes.v4.1.sites.chrY.vcf.bgz',   # ← replace / add paths
]

# Set to an integer (e.g. 10_000) for a quick test run; None = full file
MAX_ROWS: int | None = None

OUT_DIR = ROOT / 'data' / 'raw'
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
# ── Parse ──────────────────────────────────────────────────────────────────
frames = []

for vcf_path in VCF_FILES:
    vcf_path = Path(vcf_path)
    if not vcf_path.exists():
        print(f'[SKIP] not found: {vcf_path}')
        continue

    print(f'Parsing {vcf_path.name} ...', end=' ', flush=True)
    df = parse_vcf_to_dataframe(str(vcf_path), max_rows=MAX_ROWS)
    df['source_file'] = vcf_path.name
    frames.append(df)
    print(f'{len(df):,} variants')

if not frames:
    raise FileNotFoundError('No VCF files found. Check VCF_FILES above.')

variants = pd.concat(frames, ignore_index=True)
print(f'\nTotal variants: {len(variants):,}')
variants.head()

Parsing gnomad.genomes.v4.1.sites.chrY.vcf.bgz ... 

1,169,063 variants

Total variants: 1,169,063


,chrom,pos,ref,alt,qual,filter_status,af_global,af_afr,af_eur,af_eas,af_amr,af_sas,ac,an,variant_type,source_file
0,chrY,2781489,C,T,None,AC0,0.0,0.0,0.0,0.0,0.0,0.0,0,2716,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
1,chrY,2781499,C,A,None,AC0,0.0,0.0,0.0,0.0,0.0,0.0,0,4097,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
2,chrY,2781511,C,G,None,AC0,0.0,0.0,0.0,0.0,0.0,0.0,0,5637,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
3,chrY,2781514,C,A,None,AC0;AS_VQSR,0.0,0.0,0.0,0.0,0.0,0.0,0,6022,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
4,chrY,2781520,G,C,None,AC0;AS_VQSR,0.0,0.0,0.0,0.0,0.0,0.0,0,6808,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz


In [12]:
# ── Save ───────────────────────────────────────────────────────────────────
out_path = ROOT / 'data' / 'raw'  / 'variants.parquet'
variants.to_parquet(out_path, index=False, engine='pyarrow')
print(f'Saved → {out_path}')

Saved → /mnt/c/IISER/SEM_8/ds_practice/genomic-variation-landscape-and-population-comparison-main/genomic-variation-landscape-and-population-comparison-main/data/raw/variants.parquet


In [13]:
variants.tail()

,chrom,pos,ref,alt,qual,filter_status,af_global,af_afr,af_eur,af_eas,af_amr,af_sas,ac,an,variant_type,source_file
1169058,chrY,56887853,G,T,None,AC0,0.0,0.0,0.0,0.0,0.0,0.0,0,594,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
1169059,chrY,56887855,T,C,None,AC0;AS_VQSR,0.0,0.0,0.0,0.0,0.0,0.0,0,549,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
1169060,chrY,56887856,T,A,None,AC0;AS_VQSR,0.0,0.0,0.0,0.0,0.0,0.0,0,519,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
1169061,chrY,56887899,T,C,None,AC0;AS_VQSR;InbreedingCoeff,NaN,NaN,NaN,NaN,NaN,NaN,0,0,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
1169062,chrY,56887900,T,TCATGTG,None,AC0;AS_VQSR;InbreedingCoeff,NaN,NaN,NaN,NaN,NaN,NaN,0,0,INDEL,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
